# Tutorial showing usage of pretrained autoencoder models 
This tutorial focus on the usage of pretrained models for scRNASeq data and provided under: 
https://huggingface.co/collections/autoencodix/acx-pretrained-models 

### Download the model files
In this tutorial we will use the a Ontix model with explainable latent space based on ontologies generated with Gemini3ProPreview:

https://huggingface.co/autoencodix/Ontix-Dim24-Gemini3ProPreview



In [ ]:
import os
from huggingface_hub import snapshot_download

model_name = "Ontix-Dim24-Gemini3ProPreview"
repo_id = f"autoencodix/{model_name}"
target_folder = f"./acx_pretrained_models/{model_name}"
private = True  # Set to True if the repository is private
if private:
	token = os.environ.get("HF_TOKEN")
	if token is None:
		raise EnvironmentError(
			"HF_TOKEN is not set."
		)
else:
	token = None

local_dir = snapshot_download(
    repo_id=repo_id,
    local_dir=target_folder,
    local_dir_use_symlinks=False,
    token=token,  # uncomment if repo is private
)

print(f"All repo files downloaded to: {local_dir}")

## Download and prepare example scRNASeq data
To show how to use scRNASeq with pretrained models, we use an example lung tissue dataset originally extracted from the [CZ CELLxGENE Census](https://chanzuckerberg.github.io/cellxgene-census/) database. It is hosted on Hugging Face Hub ([autoencodix/census-lung](https://huggingface.co/datasets/autoencodix/census-lung)) and is downloaded automatically in the cell below on first run — no manual download needed.

As an example we will select a specific cell type (respiratory basal cell) from lung tissue for both normal samples and from samples with cystic fibrosis

In [ ]:
import pandas as pd

# Get the gene space of the autoencoder to subset to only the genes it was trained on
ont_file = f"./acx_pretrained_models/{model_name}/ontology/Dim24_Gemini3ProPreview_ontology_task__ensembl_level2.tsv"
ont_features = pd.read_csv(ont_file, sep='\t', usecols=[0], header=None)
ont_features.columns = ['feature_id']

# Define the observation filter to get only the relevant cell types, tissues, and diseases
obs_value_filter = "tissue_general == 'lung' and disease in ['cystic fibrosis', 'normal'] and cell_type in ['respiratory basal cell'] and is_primary_data == True"

In [ ]:
from huggingface_hub import hf_hub_download
import anndata as ad

census_path = hf_hub_download(
    repo_id="autoencodix/census-lung", repo_type="dataset", filename="census_lung.h5ad"
)
adata = ad.read_h5ad(census_path)
adata = adata[adata.obs.query(obs_value_filter).index, adata.var.feature_id.isin(ont_features.feature_id)].copy()

adata

Importantly, models have been trained on log1p normalized counts. We do this manually and then create a data package for autoencodix:

In [ ]:
import scanpy 
from autoencodix.data._numeric_dataset import NumericDataset
from autoencodix.data._datasetcontainer import DatasetContainer
from autoencodix.configs.ontix_config import OntixConfig

scanpy.pp.log1p(adata, copy=False)

test_dataset = NumericDataset(
			data=adata.X, # the normalized expression data per cell
			config=OntixConfig(), # Empty placeholder
			sample_ids=adata.obs.index, # cell ids
			metadata=adata.obs.loc[adata.obs.index,:], # cell annotation data for plotting etc.
			split_indices=None, # Empty placeholder
			feature_ids=adata.var.feature_id, # Gene ids
		)

acx_container = DatasetContainer(train=None, valid=None, test=test_dataset)

# Save the container 
import pickle
# Create directory
os.makedirs("./TutData", exist_ok=True)

with open(f"./TutData/scRNASeq_cystic_fibrosis_{model_name}.pkl", "wb") as f:
	pickle.dump(acx_container, f)

## Calculate embeddings using a pretrained model
Now we can load the data and a model to calculate embeddings and visualize them

In [ ]:
model_name = "Ontix-Dim24-Gemini3ProPreview"
import autoencodix as acx
import pickle

ontix_file_path = f"./acx_pretrained_models/{model_name}/large_ontix_final_model_Dim24_Gemini3ProPreview_ontology_task__.pkl"

print("Loading trained model ...")
loaded_ontix = acx.Ontix.load(file_path=ontix_file_path)
loaded_ontix._trainer._config.save_vram = True  # Enable memory saving for prediction

print("Load test data container ...")
with open(f"./TutData/scRNASeq_cystic_fibrosis_{model_name}.pkl", "rb") as f:
	acx_container = pickle.load(f)

print("Calculate embeddings ...")
result = loaded_ontix.predict(data=acx_container)


In [ ]:
# Create a latent space visualization
loaded_ontix.visualizer.show_latent_space(
	result=result,
	plot_type="Ridgeline",
	param=['disease'], 
	split="test",
)

## Using explain() functionality for xAI
We can leverage posthoc xAI and LLMs to get a better understanding which genes and processes drive differences of classes like cystic fibrosis vs. normal cells

For this we will firstly determine which latent dimensions show highest class separation: 

In [ ]:
# Perform Linear Discriminant Analysis (LDA) to find the latent dimension that best separates the groups
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import cross_validate

import numpy as np
import pandas as pd

anno_col = "disease"
groups_list = [
	"cystic fibrosis",
	"normal"
]
# Get the latent space as df
df_latent = loaded_ontix.result.get_latent_df(split="test", epoch=-1)

scores = {}
# for each latent dimension
for latent_dim in df_latent.columns:
	lda = LinearDiscriminantAnalysis()
	X = df_latent[latent_dim].values.reshape(-1, 1)
	y = loaded_ontix.result.new_datasets.test.metadata[anno_col].values
	# Reduce x and y to only include samples from the groups of interest
	mask = np.isin(y, groups_list)
	X = X[mask]
	y = y[mask]
	metric = "roc_auc_ovo"
 	# Do CV-5 fold cross-validation and calculate the AUC-ROC score for this latent dimension
	scores[latent_dim] = cross_validate(lda, X, y, cv=5, scoring=metric, return_train_score=True, n_jobs=-1)

scores_df = pd.DataFrame({
	'latent_dim': list(scores.keys()),
	'test_score_mean': [scores[ld]['test_score'].mean() for ld in scores.keys()],
	'test_score_std': [scores[ld]['test_score'].std() for ld in scores.keys()],
	'train_score_mean': [scores[ld]['train_score'].mean() for ld in scores.keys()],
	'train_score_std': [scores[ld]['train_score'].std() for ld in scores.keys()],
})
scores_df = scores_df.sort_values(by='test_score_mean', ascending=False)
scores_df

The highest seperation of cystic fibrosis vs normal is observed for `dim12_lipid_membrane_dynamics` and `dim20_glucose_homeostasis`. Now let's calculatue the feature (gene) attribution to those dimensions and identify potential marker genes and get a functional analyses via an LLM.

To use the openrouter API store make yor API key available as environment variable `OPENROUTER_PREMIUM_API_KEY`. Accordingly, using SCADS-LLM server with API key under `SCADS_LLM_API_KEY`. See also `LLM_Setup.md`.

In [ ]:
selected_dim = list(scores_df.iloc[0:2]['latent_dim'].values) # Top latent dimensions with highest AUC-ROC score for separating the groups
top_n_genes = 15 # Number of top contributing genes per latent dimension

latent_contributions = loaded_ontix.explain(
    split="test",
    method="IntegratedGradients", # "DeepLiftShap" or "IntegratedGradients"
    n_subset=400, # Randomly sample in input and baseline space for computational efficiency
    sel_latent_dim=selected_dim, # Or specify a single latent dimension to explain    
	input_type="grouped", # We will test group 'cystic fibrosis' (input) vs. 'normal' (baseline)
	input_group=groups_list[0], # "cystic fibrosis" 
	baseline_type="random",
	baseline_group=groups_list[1], # "normal"
	anno_col=anno_col,
	llm_explain=True, 	# Explain biological functions of the contributing genes using a LLM
	llm_client="openrouter",
	llm_model="google/gemma-4-31b-it",
	# llm_client="scads-llm",
	# llm_model="google/gemma-4-31b-it",
	# llm_model="alias-ha",
	top_n_genes=top_n_genes,
)

The LLM-based verbal explaination and summary is stored as a markdown file:

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display

md_path = Path("latent_explanations.md")
display(Markdown(md_path.read_text(encoding="utf-8")))

## Using the .generate() functionality for synthetic data generation
Leveraging our pretrained models we can sample from the latent space artificial cells. 

We can do this either randomly across the whole latent space or from pre-defined points of the latent space. 

We will do here the latter by calculating the mean of our cystic fibrosis cells for each latent dimension and generate 1000 synthetic cells from this point of the latent space.

In [ ]:
df_latent_cf = df_latent.loc[
	result.new_datasets.test.metadata.disease == 'cystic fibrosis',
 :
]

latent_prior_means = df_latent_cf.mean()

# Generate 1000 latent prior samples based on means and some noise
num_samples = 1000
noise = np.random.normal(0, 0.1, (num_samples, len(latent_prior_means)))
synthetic_latent = np.tile(latent_prior_means.values, (num_samples, 1)) + noise

In [ ]:
# Generate reconstructions (gene expression) from the synthetic latent samples
generated_reconstructions = loaded_ontix.generate(latent_prior=synthetic_latent)
print("Generated reconstructions shape:", generated_reconstructions.shape)